# Training GPT from Scratch

End-to-end pipeline:
1. Load raw text data
2. Tokenise with **tiktoken** (`gpt2` encoding — no training needed)
3. Build a sliding-window `Dataset` + `DataLoader`
4. Configure and instantiate the `GPT` model
5. Train with `Trainer`
6. Save the model
7. Test generation

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

from GPT.config import GPTConfig
from GPT.GPT import GPT
from GPT.trainer import Trainer
from utils.utility import device_setup

device = device_setup(seed=42)
print(f"\nDevice: {device}")

W0617 23:55:06.345000 28436 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W0617 23:55:06.781000 28436 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\Jack\anaconda3\envs\vit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: NVIDIA GeForce RTX 2070 SUPER
cuDNN benchmark: True
cuDNN deterministic: True

Device: cuda


## 1. Load Training Data

In [3]:
TRAIN_PATH = "data/train.txt"
VAL_PATH   = "data/validation.txt"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_text = f.read()
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_text = f.read()

print(f"Train : {len(train_text):,} chars")
print(f"Val   : {len(val_text):,} chars")
print(f"\nFirst 300 chars:\n{train_text[:300]!r}")

Train : 1,931,768,988 chars
Val   : 19,520,168 chars

First 300 chars:
'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and '
